# 🎙️ BookVoice-AI — GPU Server (XTTS-v2)
**Google Colab T4 GPU · Türkisch/Deutsch/Englisch · Voice Cloning**

📖 [GitHub](https://github.com/dolunay38/BookVoice-AI) | 🌐 [ahrar.aksoy-net.de](https://ahrar.aksoy-net.de)

---
**Reihenfolge:**
1. ▶️ Schritt 1 — Einmalig (beim ersten Mal ~15 Min)
2. ▶️ Schritt 2 — Bei jedem Neustart
3. ▶️ Schritt 3 — IMMER laufen lassen!

In [ ]:
#@title ⚙️ Schritt 1: Installation (nur beim ersten Mal)
import subprocess, sys, os

print('📦 Schritt 1: Pakete installieren...')
os.system('pip install -q transformers==4.45.0')
os.system('pip install -q coqui-tts faster-whisper httpx')
print('📦 Pakete OK!')

print('📦 Schritt 2: PyTorch Audio...')
os.system('pip install -q torchaudio --index-url https://download.pytorch.org/whl/cu118')
print('📦 PyTorch OK!')

print('📦 Schritt 3: XTTS-v2 Modell (~1.8 GB)...')
os.environ['COQUI_TOS_AGREED'] = '1'

# Transformers Compatibility Fix
import transformers.pytorch_utils as _pu
import torch as _torch
if not hasattr(_pu, 'isin_mps_friendly'):
    _pu.isin_mps_friendly = _torch.isin

from TTS.api import TTS
TTS('tts_models/multilingual/multi-dataset/xtts_v2')
print('✅ Alles bereit! Jetzt Schritt 2 ausführen.')

In [ ]:
#@title 🚀 Schritt 2: GPU Server starten (bei jedem Neustart)
#@markdown Klicke auf ▶ — warte bis die URL erscheint, dann kopieren!
import subprocess, os, time, re, urllib.request

# Transformers Compatibility Fix
import transformers.pytorch_utils as pu
import torch
if not hasattr(pu, 'isin_mps_friendly'):
    pu.isin_mps_friendly = torch.isin

# tts_server.py von GitHub laden
print('📥 Lade tts_server.py von GitHub...')
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/dolunay38/BookVoice-AI/main/tts_server.py',
    'tts_server.py'
)
print('✅ tts_server.py geladen!')

# Ordner erstellen
os.makedirs('/content/HOERBUCH', exist_ok=True)
os.makedirs('/content/TRANSKRIPTIONEN', exist_ok=True)

# Alte Prozesse beenden
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

# Server starten
env = os.environ.copy()
env['TTS_OUTPUT'] = '/content/HOERBUCH'
env['TTS_LANG'] = 'tr'
env['COQUI_TOS_AGREED'] = '1'

server = subprocess.Popen(
    ['uvicorn', 'tts_server:app', '--host', '0.0.0.0', '--port', '7500'],
    env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
print('⏳ Server startet...')
time.sleep(10)

# Health Check
try:
    resp = urllib.request.urlopen('http://localhost:7500/health', timeout=5)
    print('✅ Server läuft!')
except:
    print('⚠️ Server braucht noch etwas...')
    time.sleep(5)

# Cloudflared Tunnel
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/usr/local/bin/cloudflared'
], capture_output=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:7500'],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
)

print('⏳ Warte auf URL...')
for i in range(60):
    line = proc.stderr.readline()
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            print('='*50)
            print('🎉 BOOKVOICE-AI GPU BEREIT!')
            print(f'🌐 URL: {url}')
            print('='*50)
            print('👆 URL kopieren → BookVoice-AI GUI → GPU Colab → Verbinden')
            break
    time.sleep(1)

In [ ]:
#@title ⏳ Schritt 3: Session aktiv halten — IMMER laufen lassen!
#@markdown Diese Zelle verhindert dass Colab die Session trennt.
import time
print('✅ Session aktiv — nicht stoppen!')
print('💡 Du kannst jetzt BookVoice-AI im Browser nutzen.')
counter = 0
while True:
    time.sleep(60)
    counter += 1
    if counter % 10 == 0:
        print(f'⏳ Aktiv seit {counter} Minuten...')

In [ ]:
#@title 📥 Optional: Hörbücher herunterladen
from google.colab import files
import os, glob

hoerbuecher = glob.glob('/content/HOERBUCH/**/*.mp3', recursive=True)
hoerbuecher += glob.glob('/content/HOERBUCH/**/*.m4b', recursive=True)

if hoerbuecher:
    print(f'📚 {len(hoerbuecher)} Hörbücher gefunden:')
    for f in hoerbuecher:
        print(f'  - {os.path.basename(f)}')
    
    datei = hoerbuecher[-1]  # Letzte Datei
    print(f'\n⬇️ Lade herunter: {os.path.basename(datei)}')
    files.download(datei)
else:
    print('Noch keine Hörbücher generiert.')